# MNIST MLP3 — SGD + momentum + Muon baseline

This is a **clean optimizer baseline** with three independent seeds and two-sided 95% Student-t confidence intervals. It contains no RG intervention.

At epoch 0 and after every epoch, WeightWatcher runs

```python
watcher.analyze(ERG=True, randomize=True)
```

so every FC1/FC2/FC3 checkpoint records `alpha`, the randomized-MP correlation-trap count `num_traps`, `detX_num`, `num_pl_spikes`, `ERG_gap`, midpoint rank, and midpoint trace-log. The trap plot is saved as `7_layerwise_weightwatcher_num_traps_95ci.png`.

Optimizer definition: Muon/Newton--Schulz on `fc1.weight` and `fc2.weight`; SGD + momentum on `fc3.weight` and all biases.


In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

MIN_WW_VERSION = (0, 7, 7)

def version_tuple(value):
    parts = []
    for token in str(value).split("."):
        digits = "".join(character for character in token if character.isdigit())
        if not digits:
            break
        parts.append(int(digits))
    return tuple((parts + [0, 0, 0])[:3])

needs_install = False
try:
    weightwatcher = importlib.import_module("weightwatcher")
    needs_install = (
        version_tuple(getattr(weightwatcher, "__version__", "0")) < MIN_WW_VERSION
    )
except ImportError:
    needs_install = True

if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "weightwatcher>=0.7.7",
    ])
    for module_name in list(sys.modules):
        if module_name == "weightwatcher" or module_name.startswith("weightwatcher."):
            del sys.modules[module_name]
    importlib.invalidate_caches()
    weightwatcher = importlib.import_module("weightwatcher")
    if version_tuple(getattr(weightwatcher, "__version__", "0")) < MIN_WW_VERSION:
        raise RuntimeError("WeightWatcher upgrade did not provide num_traps support")

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / "baseline"
    if (candidate / "rg_baselines").is_dir():
        ROOT = candidate
        break
    if (path / "rg_baselines").is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError("Run this notebook from a clone of CalculatedContent/rg_optimizers.")
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def resolve_artifact_dir(environment_variable, default):
    raw = os.environ.get(environment_variable)
    path = Path(raw).expanduser() if raw else default
    if not path.is_absolute():
        path = Path.cwd() / path
    return path.resolve()

RUN_ROOT = resolve_artifact_dir("RG_BASELINE_RUN_ROOT", ROOT / "runs")
DATA_DIR = resolve_artifact_dir("RG_BASELINE_DATA_DIR", ROOT / "data")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("baseline root:", ROOT)
print("shared run root:", RUN_ROOT)
print("MNIST data directory:", DATA_DIR)


In [ ]:
from dataclasses import asdict
from IPython.display import display
import numpy as np
import pandas as pd

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    plot_all_replicates,
    run_baseline_replicates,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 1000)
SEEDS = DEFAULT_BASELINE_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3

CONFIG = BaselineConfig(
    optimizer="sgd_momentum_muon",
    epochs=20,
    train_eval_max_batches=None,
    strict_metrics=True,
    ww_randomize=True,
    muon_parameter_names=("fc1.weight", "fc2.weight"),
    muon_learning_rate=0.02,
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_newton_schulz_steps=5,
    muon_aux_learning_rate=0.05,
    muon_aux_momentum=0.9,
    muon_aux_dampening=0.0,
    muon_aux_nesterov=False,
    muon_aux_weight_decay=1e-4,
    save_epoch_checkpoints=True,
)
RUN_DIR = RUN_ROOT / CONFIG.run_slug
PLOT_DIR = RUN_DIR / "plots"
print("Independent seeds:", SEEDS)
print("run directory:", RUN_DIR)
print("randomized correlation-trap analysis:", CONFIG.ww_randomize)
display(pd.DataFrame([asdict(CONFIG)]))


In [ ]:
from rg_baselines import MLP3, build_optimizer
probe_model = MLP3()
probe_optimizer = build_optimizer(probe_model, CONFIG)
assignment = getattr(probe_optimizer, "assignment", {})
display(pd.DataFrame([
    {"parameter": name, "optimizer_path": path} for name, path in assignment.items()
]).sort_values("parameter"))
assert assignment["fc1.weight"] == "muon"
assert assignment["fc2.weight"] == "muon"
assert assignment["fc3.weight"] == "sgd"


In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
)

plot_all_replicates(suite, output_dir=PLOT_DIR, show=True)

expected_paths = [
    RUN_DIR / "performance_by_epoch_and_seed.csv",
    RUN_DIR / "spectral_metrics_by_epoch_layer_and_seed.csv",
    RUN_DIR / "performance_summary_95ci.csv",
    RUN_DIR / "spectral_summary_95ci.csv",
    RUN_DIR / "replicate_manifest.json",
    PLOT_DIR / "7_layerwise_weightwatcher_num_traps_95ci.png",
]
for seed in SEEDS:
    seed_dir = RUN_DIR / "seeds" / f"seed_{seed}"
    expected_paths.extend([
        seed_dir / "performance_by_epoch.csv",
        seed_dir / "spectral_metrics_by_epoch_and_layer.csv",
        seed_dir / "esd_history.npz",
        seed_dir / "config.json",
        seed_dir / "final_state.pt",
    ])
    expected_paths.extend(
        seed_dir / "checkpoints" / f"epoch_{epoch:03d}.pt"
        for epoch in range(1, CONFIG.epochs + 1)
    )
missing = [path for path in expected_paths if not path.is_file()]
if missing:
    raise RuntimeError(
        "The baseline run completed but required persisted artifacts are missing:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )
print("saved aggregate results:", RUN_DIR)
print("saved plots:", PLOT_DIR)
print("verified persisted files:", len(expected_paths))


## Performance per epoch

In [ ]:
performance_columns = [
    "epoch", "metric", "n", "mean", "std", "sem",
    "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
]
display(
    suite.performance_summary.loc[
        suite.performance_summary["metric"].isin(
            ["train_loss", "test_loss", "train_accuracy", "test_accuracy"]
        ),
        performance_columns,
    ].sort_values(["metric", "epoch"])
)


## WeightWatcher metrics, including randomized correlation traps

`num_traps` is taken directly from `watcher.analyze(randomize=True)`; no proxy or fallback trap count is used.

In [ ]:
required_ww_metrics = [
    "alpha",
    "num_traps",
    "detX_num",
    "num_pl_spikes",
    "ERG_gap",
    "m_midpoint",
    "trace_log_midpoint_per_eval",
    "trace_log_midpoint_total",
]
spectral_columns = [
    "layer", "epoch", "metric", "n", "mean", "std", "sem",
    "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
]
display(
    suite.spectral_summary.loc[
        suite.spectral_summary["metric"].isin(required_ww_metrics),
        spectral_columns,
    ].sort_values(["metric", "layer", "epoch"])
)


## Additional spectral diagnostics

In [ ]:
additional_spectral_metrics = [
    "stable_rank", "participation_ratio", "entropy_effective_rank",
    "boundary_overlap_ratio", "top1_energy_fraction", "pl_energy_fraction",
    "detx_energy_fraction", "midpoint_energy_fraction",
    "geometric_mean_midpoint", "normalized_lambda_max",
    "normalized_lambda_midpoint_cut", "eigenvalue_condition_number",
]
display(
    suite.spectral_summary.loc[
        suite.spectral_summary["metric"].isin(additional_spectral_metrics),
        spectral_columns,
    ].sort_values(["metric", "layer", "epoch"])
)


display(
    suite.performance_summary.loc[
        suite.performance_summary["metric"].isin(
            [
                "mean_gradient_norm_before_clip",
                "max_gradient_norm_before_clip",
                "parameter_l2_norm",
                "train_time_sec",
                "evaluation_time_sec",
                "weightwatcher_time_sec",
            ]
        ),
        performance_columns,
    ].sort_values(["metric", "epoch"])
)


In [ ]:
expected_epochs = set(range(CONFIG.epochs + 1))
assert set(suite.performance["epoch"].astype(int)) == expected_epochs
assert set(suite.performance["seed"].astype(int)) == set(SEEDS)
valid = suite.spectral_metrics.loc[suite.spectral_metrics["status"].eq("ok")]
assert "num_traps" in valid.columns
assert np.isfinite(valid["num_traps"].to_numpy(dtype=float)).all()
assert (valid["num_traps"].to_numpy(dtype=float) >= 0).all()
assert np.allclose(
    valid["num_traps"].to_numpy(dtype=float),
    np.rint(valid["num_traps"].to_numpy(dtype=float)),
)
for epoch in expected_epochs:
    for layer in ("fc1", "fc2", "fc3"):
        observed = set(
            valid.loc[
                valid["epoch"].eq(epoch) & valid["layer"].eq(layer), "seed"
            ].astype(int)
        )
        assert observed == set(SEEDS), (epoch, layer, observed)
for seed in SEEDS:
    checkpoint_dir = RUN_DIR / "seeds" / f"seed_{seed}" / "checkpoints"
    saved_epochs = {
        int(path.stem.split("_")[-1]) for path in checkpoint_dir.glob("epoch_*.pt")
    }
    assert saved_epochs == set(range(1, CONFIG.epochs + 1))
print(
    "Audit passed:", len(SEEDS), "seeds;", CONFIG.epochs + 1,
    "metric checkpoints; FC1/FC2/FC3 alpha, num_traps, ERG and trace-log at every epoch."
)
